In [1]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '1'

from cmm_eval import BatchRecorder, BatchStore, CMMSaturationEvaluator, default_checkpoints, plot_sweep

import torch
from MeMoHF.modelling_memo_tokenizer import MeMoTokenizer
from MeMoHF.modelling_memo_configuration import MeMoConfig
from MeMoHF.modelling_memo import MeMoForCausalLM
from MeMoHF.evaluating_memo import Evaluation
from MeMoHF.utils import seed_everything

seed_everything(42)

In [2]:
d,h,l = 2048, 4, 5
chunk_length =  (h ** l)

compositionOp='SCC'#'JLT' #'ProdWithShuffling' #

NUMBER_OF_DOCS=100

CLEN_AFTER_RUN = True

In [3]:
padding_vector_component_values= {
    'ProdWithShuffling': 1/(d**(1/2)),
    'Prod': 1/(d**(1/2)), 
    'Sum': 0,
    'SCC': 'circular_convolution_identity',
    'JLT': 'in_distribution'
}
padding_vector_component_values = padding_vector_component_values[compositionOp]


# Initializing a standard Tokenizer
max_length = chunk_length 
tokenizer = MeMoTokenizer.from_pretrained("EleutherAI/gpt-neox-20b", 
                                          padding_side='left', truncation_side='left', 
                                          model_max_length=max_length, head_number=h)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.pad_token_id = tokenizer.pad_token_id

# Intializing Memo Configuration
config = MeMoConfig(vocab_size=len(tokenizer), #tokenizer.vocab_size, 
    hidden_size=d, 
    num_hidden_layers=l,
    num_attention_heads=h,
    chunk_length=chunk_length,
    bos_token_id=tokenizer.bos_token_id,
    eos_token_id=tokenizer.eos_token_id,
    pad_token_id=tokenizer.pad_token_id,
    compositionOp=compositionOp,#'ProdWithShuffling',
    padding_seq_idx=tokenizer.pad_token_id,
    padding_vector_component_values=padding_vector_component_values #new!
                    
)

# Initializing the Memo Model from the configuration

model = MeMoForCausalLM(config) 
model.training = True
model.train()
print(model)
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)} is available.")
    model.to('cuda')


The tokenizer class you load from this checkpoint is not the same type as the class this function is called from. It may result in unexpected tokenization. 
The tokenizer class you load from this checkpoint is 'GPTNeoXTokenizer'. 
The class this function is called from is 'MeMoTokenizer'.
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


Setting pad token and pad token id = <|endoftext|>, 0
MeMoEmbedding circular_convolution_identity
MeMoEmbedding 0
MeMo embedding initilialization:  MeMoEmbedding
Padding vector components :  circular_convolution_identity
MeMo embedding initilialization:  MeMoEmbedding
Padding vector components :  0
MeMo embedding initilialization:  MeMoEmbedding
Padding vector components :  circular_convolution_identity
MeMo embedding initilialization:  MeMoEmbedding
Padding vector components :  0
MeMo embedding initilialization:  MeMoEmbedding
Padding vector components :  0
MeMoForCausalLM(
  (memo): MeMo(
    (encoder): MeMoEmbedding(50277, 2048, padding_idx=0)
    (output_encoder): MeMoEmbedding(50277, 2048, padding_idx=0)
    (layers): MeMoLayers(
      (0-4): 5 x MeMoLayer(
        (CMM_OUT): CorrelationMatrixMemory(in_features=2048, out_features=2048)
      )
    )
  )
  (lm_head): MeMoEmbedding(50277, 2048, padding_idx=0)
)
GPU: NVIDIA RTX A6000 is available.


In [4]:
from cmm_lda import run_lda_experiment

In [5]:
import copy
from pathlib import Path

import matplotlib.pyplot as plt
import torch

from cmm_eval import (BatchRecorder, BatchStore, CMMSaturationEvaluator,
                      default_checkpoints, plot_sweep)


class CMM_evaluator:
    """
    Launches the CMM saturation experiments.

        exp = CMM_evaluator(model, tokenizer, "memo_dump")
        exp.record(data)                 # once: runs model.memorize_text and dumps every layer's memorize() calls
        exp.evaluate(2)                  # one layer
        exp.evaluate_all()               # every layer

    In a new process (model rebuilt from scratch), call exp.load_layer_states() before evaluating, so that the
    random projections / shuffling buffers are the ones used when recording.
    """

    def __init__(self, model, tokenizer, dump_dir):
        self.model = model
        self.tokenizer = tokenizer  # not used here, kept for compatibility
        self.dump_dir = Path(dump_dir)

    # ------------------------------------------------------------------ #
    # recording
    # ------------------------------------------------------------------ #
    def record(self, data, max_batches_per_layer=1000, save_dtype=None,
               append=False, save_layer_states=True):
        """
        Run model.memorize_text on every item of `data` while recording each layer's memorize() calls.
        max_batches_per_layer caps disk usage; save_dtype=torch.float16 halves it (replay is then lossy).
        Refuses to write into a folder that already holds batches unless append=True, because
        recording twice silently duplicates the data.
        """
        if not append and any(self.dump_dir.glob("layer_*/manifest.jsonl")):
            raise FileExistsError(
                f"{self.dump_dir} already contains recorded batches; "
                "use a new dump_dir or pass append=True")
        self.dump_dir.mkdir(parents=True, exist_ok=True)

        rec = BatchRecorder(self.dump_dir, save_dtype=save_dtype,
                            max_batches_per_layer=max_batches_per_layer)
        for l, layer in enumerate(self.model.memo.layers):
            rec.attach(layer, l)
        try:
            for i in range(len(data)):
                self.model.memorize_text(data[i])
        finally:
            rec.detach_all()  # always restore the original memorize, even if something fails

        if save_layer_states:
            self.save_layer_states()
        return self.summary()

    def summary(self):
        """Number of recorded batches and pairs per layer."""
        out = {}
        for l in range(len(self.model.memo.layers)):
            try:
                store = BatchStore(self.dump_dir, l)
            except FileNotFoundError:
                continue
            out[l] = dict(n_batches=len(store),
                          n_pairs=sum(store.n_pairs(i) for i in range(len(store))))
        return out

    # ------------------------------------------------------------------ #
    # layer states (projections / shuffling buffers must match the recording)
    # ------------------------------------------------------------------ #
    def save_layer_states(self):
        torch.save({l: layer.state_dict() for l, layer in enumerate(self.model.memo.layers)},
                   self.dump_dir / "layers.pt")

    def load_layer_states(self):
        states = torch.load(self.dump_dir / "layers.pt", map_location="cpu")
        for l, layer in enumerate(self.model.memo.layers):
            layer.load_state_dict(states[l])

    # ------------------------------------------------------------------ #
    # evaluation
    # ------------------------------------------------------------------ #
    def evaluate(self, layer_level, n_holdout_batches=2, n_points=8, n_eval=2000, n_fresh=2000,
                 decode=True, prior_smoothing=1.0, plot=True):
        """
        Replay the recorded batches of one layer into an empty copy of that layer and measure recall
        (cosine, top-1, held-out crosstalk) and, if decode=True, the decoders (dot / Bayes temperature /
        fitted temperature, with and without prior). Returns the list of per-checkpoint rows.
        """
        # sweep() zeroes CMM_OUT, so work on a copy: the model's real memory stays untouched.
        layer = copy.deepcopy(self.model.memo.layers[layer_level])
        store = BatchStore(self.dump_dir, layer_level)

        ev = CMMSaturationEvaluator(layer, store, n_holdout_batches=n_holdout_batches)
        checkpoints = default_checkpoints(ev.max_train_batches, n_points=n_points)

        rows = ev.sweep(checkpoints, n_eval=n_eval, n_fresh=n_fresh,
                        out_json=str(self.dump_dir / f"layer{layer_level}_sweep.json"),
                        decode=decode, prior_smoothing=prior_smoothing)
        if plot:
            print(f"Plotting layer {layer_level} sweep to {self.dump_dir / f'layer{layer_level}_sweep.png'}")
            fig = plot_sweep(rows, str(self.dump_dir / f"layer{layer_level}_sweep.png"))
            plt.close(fig)

        del ev, layer
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        return rows

    def evaluate_all(self, layers=None, **kwargs):
        """evaluate() on every layer (or on `layers`); kwargs are passed through. Returns {layer: rows}."""
        layers = range(len(self.model.memo.layers)) if layers is None else layers
        return {l: self.evaluate(l, **kwargs) for l in layers}

    def lda(self, layer_level, n_train, n_holdout_batches=2, **kwargs):
        layer = self.model.memo.layers[layer_level]
        store = BatchStore(self.dump_dir, layer_level)
        ev = CMMSaturationEvaluator(layer, store, n_holdout_batches=n_holdout_batches)
        return run_lda_experiment(ev, n_train, **kwargs)

In [6]:
format(NUMBER_OF_DOCS, '06d')

'000100'

In [7]:
from datasets import Dataset, DatasetDict, Features, Value, load_dataset, load_from_disk, concatenate_datasets

data = load_from_disk(f'LearningEvaluation/training_data/scaling_curve/n={format(NUMBER_OF_DOCS, "06d")}') #original_training_data

tok_data = data.map(lambda x: tokenizer.get_text_batch_encoding(x['text'], truncation=True))
tok_data.set_format(type='torch', columns=['input_ids', 'labels'])
print(tok_data)
tok_data[0]['input_ids'].shape, tok_data[0]['labels'].shape

Dataset({
    features: ['text', 'input_ids', 'labels'],
    num_rows: 100
})


(torch.Size([1, 1024]), torch.Size([1, 1024]))

In [8]:
from torch import exp


dump_dir = f'memo_dump_v4_{compositionOp}_{h}|{l}|{d}_{format(NUMBER_OF_DOCS, "06d")}'


cmm_evaluator = CMM_evaluator(
    model=model, 
    tokenizer=tokenizer, 
    dump_dir=dump_dir
)

cmm_evaluator.record(tok_data)

{0: {'n_batches': 100, 'n_pairs': 102400},
 1: {'n_batches': 100, 'n_pairs': 102000},
 2: {'n_batches': 100, 'n_pairs': 100800},
 3: {'n_batches': 100, 'n_pairs': 96000},
 4: {'n_batches': 100, 'n_pairs': 76800}}

In [9]:
lda_results = {}
for layer_level in range(l):
    lda_results[layer_level] = cmm_evaluator.lda(layer_level=layer_level, n_train=NUMBER_OF_DOCS)

/home/elena/MeMo/LearningEvaluation/cmm_eval.py:278: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  x = torch.load(self.dir / f"batch_{i:06d}.vals.pt", map_location="cpu")
/h

[composed] random         m= 2048 seen=1.00  cos=0.103  decode=0.093  centroid=0.554  cos(own/other centroid)=0.277/0.000  (chance 6.8e-05)
[composed] identity       m= 2048 seen=1.00  cos=0.140  decode=0.145  centroid=0.598  cos(own/other centroid)=0.277/0.000  (chance 6.8e-05)
[composed] lda_shrink0.1  m= 2048 seen=1.00  cos=0.147  decode=0.161  centroid=0.606  cos(own/other centroid)=0.279/-0.000  (chance 6.8e-05)
[composed] lda_shrink0.5  m= 2048 seen=1.00  cos=0.145  decode=0.156  centroid=0.603  cos(own/other centroid)=0.278/-0.000  (chance 6.8e-05)
[concat] random         m= 2048 seen=1.00  cos=0.124  decode=0.096  centroid=0.297  cos(own/other centroid)=0.361/0.013  (chance 6.8e-05)
[concat] lda_shrink0.1  m= 2048 seen=1.00  cos=0.276  decode=0.286  centroid=0.435  cos(own/other centroid)=0.416/-0.000  (chance 6.8e-05)
[concat] lda_shrink0.5  m= 2048 seen=1.00  cos=0.248  decode=0.238  centroid=0.390  cos(own/other centroid)=0.399/0.000  (chance 6.8e-05)
[composed] random      

In [10]:
import pandas as pd
for layer_level in range(l):
    print(layer_level)
    for c in ['concat', 'composed']:
        print(c)
        display(pd.DataFrame(lda_results[layer_level][c]))

0
concat


,random,lda_shrink0.1,lda_shrink0.5
m,2048.000000,2048.000000,2048.000000
n_train_pairs,102400.000000,102400.000000,102400.000000
frac_seen,1.000000,1.000000,1.000000
cos_mean_seen,0.123790,0.276277,0.247817
decode_acc_seen,0.096494,0.285625,0.238418
centroid_acc_seen,0.296836,0.434629,0.389785
cos_own_centroid,0.361390,0.416188,0.398725
cos_other_centroids,0.012500,-0.000003,0.000013
chance,0.000068,0.000068,0.000068


composed


,random,identity,lda_shrink0.1,lda_shrink0.5
m,2048.000000,2048.000000,2048.000000,2048.000000
n_train_pairs,102400.000000,102400.000000,102400.000000,102400.000000
frac_seen,1.000000,1.000000,1.000000,1.000000
cos_mean_seen,0.102882,0.140203,0.147329,0.145111
decode_acc_seen,0.092920,0.145127,0.160928,0.155771
centroid_acc_seen,0.554473,0.597734,0.605703,0.603369
cos_own_centroid,0.277296,0.277336,0.279256,0.278342
cos_other_centroids,0.000007,0.000007,-0.000020,-0.000019
chance,0.000068,0.000068,0.000068,0.000068


1
concat


,random,lda_shrink0.1,lda_shrink0.5
m,2048.000000,2048.000000,2048.000000
n_train_pairs,102000.000000,102000.000000,102000.000000
frac_seen,1.000000,1.000000,1.000000
cos_mean_seen,0.129181,0.225314,0.220593
decode_acc_seen,0.140853,0.394471,0.375931
centroid_acc_seen,0.605735,0.705676,0.695774
cos_own_centroid,0.270878,0.365662,0.360892
cos_other_centroids,0.000051,-0.000027,-0.000026
chance,0.000068,0.000068,0.000068


... (10 outputs left out)

4
concat


,random,lda_shrink0.1,lda_shrink0.5
m,2048.000000,2048.000000,2048.000000
n_train_pairs,76800.000000,76800.000000,76800.000000
frac_seen,1.000000,1.000000,1.000000
cos_mean_seen,0.068914,0.249614,0.239213
decode_acc_seen,0.060951,0.478555,0.434036
centroid_acc_seen,0.463854,0.744661,0.741589
cos_own_centroid,0.504418,0.387748,0.378494
cos_other_centroids,0.193624,-0.000010,-0.000006
chance,0.000079,0.000079,0.000079


composed


,random,identity,lda_shrink0.1,lda_shrink0.5
m,2048.000000,2.048000e+03,2048.000000,2048.000000
n_train_pairs,76800.000000,7.680000e+04,76800.000000,76800.000000
frac_seen,1.000000,1.000000e+00,1.000000,1.000000
cos_mean_seen,0.114444,1.607309e-01,0.161792,0.161503
decode_acc_seen,0.113398,2.499870e-01,0.258411,0.259570
centroid_acc_seen,0.620859,6.790755e-01,0.681693,0.680911
cos_own_centroid,0.288247,2.882373e-01,0.290553,0.289460
cos_other_centroids,0.000002,6.320646e-07,-0.000023,-0.000023
chance,0.000079,7.915776e-05,0.000079,0.000079


In [11]:
results = cmm_evaluator.evaluate_all()

for layer_level in range(l):
    print("*"*80)
    print(layer_level)
    print("*"*80)

    display(results[layer_level])

/home/elena/MeMo/LearningEvaluation/cmm_eval.py:274: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  x = torch.load(self.dir / f"batch_{i:06d}.keys.pt", map_location="cpu")
/h

batches=    1 P=    1024 alpha=0.500  cos=0.784 (theory 0.816, p05 0.614)  top1=0.954  d_eff~2754/2048  |k|=1.00
      NLL dot=8.52 bayes=69.09 bayes+prior=69.08 fit+prior=8.63 | white=17.03 white+prior=16.99 fit_white+prior=2.11 (uniform 9.60, prior-only 7.51)
      top-k fit+prior: 1:0.954 5:0.998 10:1.000 100:1.000  |  top-k fit_white+prior: 1:0.949 5:0.998 10:1.000 100:1.000  beta_fit/theory=0.12 (white 0.12)  V=14706  PR(C)=50.3
batches=    2 P=    2048 alpha=1.000  cos=0.671 (theory 0.707, p05 0.511)  top1=0.951  d_eff~4668/2048  |k|=1.00
      NLL dot=8.38 bayes=89.10 bayes+prior=89.10 fit+prior=11.14 | white=23.23 white+prior=23.16 fit_white+prior=2.91 (uniform 9.60, prior-only 7.52)
      top-k fit+prior: 1:0.951 5:0.996 10:1.000 100:1.000  |  top-k fit_white+prior: 1:0.942 5:0.993 10:1.000 100:1.000  beta_fit/theory=0.12 (white 0.12)  V=14706  PR(C)=79.7
batches=    4 P=    4096 alpha=2.000  cos=0.558 (theory 0.577, p05 0.424)  top1=0.959  d_eff~3481/2048  |k|=1.00
      NLL 

[{'n_batches': 1,
  'P': 1024,
  'alpha': 0.5,
  'cos_mean': 0.7844977378845215,
  'cos_p05': 0.6143152117729187,
  'cos_min': 0.2244178056716919,
  'cos_theory': 0.8164965809277261,
  'rel_err': 0.8684667944908142,
  'rel_err_theory': 0.7071067811865476,
  'top1_acc': 0.9541015625,
  'top1_chance': 0.0009765625,
  'key_norm_mean': 1.0,
  'power_ratio': 3.6892378330230713,
  'd_eff_est': 2753.779541015625,
  'fro2_per_pair': 1.063240242567452,
  'dec': {'topk': [1, 5, 10, 100],
   'ridge': 0.001,
   'n_candidates': 14706,
   'n_distinct_stored': 375,
   'gamma': 1.0,
   'beta_theory': 4066.7537719994693,
   'beta_fit': 508.34420776367205,
   'beta_fit_prior': 508.34420776367205,
   'beta_theory_white': 2.0019550342130987,
   'beta_fit_white': 0.2502443790435791,
   'beta_fit_white_prior': 0.2502443790435791,
   'C_trace': 1.0081761272274044,
   'C_participation_ratio': 50.31136461459941,
   'ridge_abs': 4.92273499622756e-07,
   'nll_uniform': 9.596010852756168,
   'top1_chance': 6.7999

********************************************************************************
1
********************************************************************************


[{'n_batches': 1,
  'P': 1020,
  'alpha': 0.498046875,
  'cos_mean': 0.8177688717842102,
  'cos_p05': 0.7485051155090332,
  'cos_min': 0.3913719058036804,
  'cos_theory': 0.8170286739010644,
  'rel_err': 0.7039825320243835,
  'rel_err_theory': 0.7057243619147634,
  'top1_acc': 1.0,
  'top1_chance': 0.000980392156862745,
  'key_norm_mean': 1.0,
  'power_ratio': 3.0353400707244873,
  'd_eff_est': 2076.046872138977,
  'fro2_per_pair': 1.0007410997199468,
  'dec': {'topk': [1, 5, 10, 100],
   'ridge': 0.001,
   'n_candidates': 14675,
   'n_distinct_stored': 374,
   'gamma': 1.0,
   'beta_theory': 4082.37813106817,
   'beta_fit': 32659.025390624978,
   'beta_fit_prior': 32659.025390624978,
   'beta_theory_white': 2.0098135426889105,
   'beta_fit_white': 16.0785083770752,
   'beta_fit_white_prior': 16.0785083770752,
   'C_trace': 1.008259844062447,
   'C_participation_ratio': 50.341530304294544,
   'ridge_abs': 4.923143769836168e-07,
   'nll_uniform': 9.593900644696298,
   'top1_chance': 6.8

********************************************************************************
2
********************************************************************************


[{'n_batches': 1,
  'P': 1008,
  'alpha': 0.4921875,
  'cos_mean': 0.8204861879348755,
  'cos_p05': 0.7567946314811707,
  'cos_min': 0.6266285181045532,
  'cos_theory': 0.8186312161546413,
  'rel_err': 0.7024518251419067,
  'rel_err_theory': 0.701560760020114,
  'top1_acc': 1.0,
  'top1_chance': 0.000992063492063492,
  'key_norm_mean': 1.0,
  'power_ratio': 3.0492284297943115,
  'd_eff_est': 2065.622257232666,
  'fro2_per_pair': 1.0080935212056525,
  'dec': {'topk': [1, 5, 10, 100],
   'ridge': 0.001,
   'n_candidates': 14588,
   'n_distinct_stored': 367,
   'gamma': 1.0,
   'beta_theory': 4130.711612569277,
   'beta_fit': 4130.711425781252,
   'beta_fit_prior': 3722.8055398656434,
   'beta_theory_white': 2.0337636544190665,
   'beta_fit_white': 0.5263710554245569,
   'beta_fit_white_prior': 0.5263710554245569,
   'C_trace': 1.0083366224772876,
   'C_participation_ratio': 50.19142153705851,
   'ridge_abs': 4.923518664439881e-07,
   'nll_uniform': 9.58795455192857,
   'top1_chance': 6.8

********************************************************************************
3
********************************************************************************


[{'n_batches': 1,
  'P': 960,
  'alpha': 0.46875,
  'cos_mean': 0.8266558647155762,
  'cos_p05': 0.7628391981124878,
  'cos_min': 0.597752571105957,
  'cos_theory': 0.8251369970070347,
  'rel_err': 0.6784944534301758,
  'rel_err_theory': 0.6846531968814576,
  'top1_acc': 1.0,
  'top1_chance': 0.0010416666666666667,
  'key_norm_mean': 1.0,
  'power_ratio': 3.1170239448547363,
  'd_eff_est': 2032.3429870605469,
  'fro2_per_pair': 0.9961955265901714,
  'dec': {'topk': [1, 5, 10, 100],
   'ridge': 0.001,
   'n_candidates': 14234,
   'n_distinct_stored': 349,
   'gamma': 1.0,
   'beta_theory': 4337.94258413499,
   'beta_fit': 4337.942382812497,
   'beta_fit_prior': 22895.797280996012,
   'beta_theory_white': 2.13555787278415,
   'beta_fit_white': 17.084463119506836,
   'beta_fit_white_prior': 17.084463119506836,
   'C_trace': 1.008224994723206,
   'C_participation_ratio': 48.1714917821279,
   'ridge_abs': 4.922973607046904e-07,
   'nll_uniform': 9.563388747859216,
   'top1_chance': 7.025432

********************************************************************************
4
********************************************************************************


[{'n_batches': 1,
  'P': 768,
  'alpha': 0.375,
  'cos_mean': 0.8543195724487305,
  'cos_p05': 0.8036925196647644,
  'cos_min': 0.5147685408592224,
  'cos_theory': 0.8528028654224417,
  'rel_err': 0.6082993745803833,
  'rel_err_theory': 0.6123724356957945,
  'top1_acc': 0.9986979365348816,
  'top1_chance': 0.0013020833333333333,
  'key_norm_mean': 1.0,
  'power_ratio': 3.639169454574585,
  'd_eff_est': 2026.8821411132812,
  'fro2_per_pair': 0.9997668184967119,
  'dec': {'topk': [1, 5, 10, 100],
   'ridge': 0.001,
   'n_candidates': 12633,
   'n_distinct_stored': 295,
   'gamma': 1.0,
   'beta_theory': 5423.357384224459,
   'beta_fit': 43386.859374999985,
   'beta_fit_prior': 43386.859374999985,
   'beta_theory_white': 2.6701434159061277,
   'beta_fit_white': 0.5058975219726562,
   'beta_fit_white_prior': 0.5058975219726562,
   'C_trace': 1.0083151374327326,
   'C_participation_ratio': 45.011257891301135,
   'ridge_abs': 4.923413756995765e-07,
   'nll_uniform': 9.444067716830038,
   'to

In [12]:
import shutil

if True:
    for layer_level in range(l):
        shutil.rmtree(f"{dump_dir}/layer_{layer_level}")

In [13]:
0

0

In [ ]:
exit()